Assignment 3 

t8220103 Βασίλειος Μυλωνάς 

In [2]:
import pandas as pd 

In [3]:
choices = pd.read_csv('https://raw.githubusercontent.com/jcpeterson/choices13k/refs/heads/main/c13k_selections.csv')

In [19]:
choices.head()

,Problem,Feedback,n,Block,Ha,pHa,La,Hb,pHb,Lb,LotShapeB,LotNumB,Amb,Corr,bRate,bRate_std
0,1,True,15,2,26,0.95,-1,23,0.05,21,0,1,False,0,0.626667,0.384460
1,2,True,15,4,14,0.60,-18,8,0.25,-5,0,1,True,-1,0.493333,0.413118
2,3,True,17,4,2,0.50,0,1,1.00,1,0,1,False,0,0.611765,0.432843
3,4,True,18,3,37,0.05,8,87,0.25,-31,1,2,False,0,0.222222,0.387383
4,5,False,15,1,26,1.00,26,45,0.75,-36,2,5,False,0,0.586667,0.450185


In [4]:
problems = pd.read_json("https://raw.githubusercontent.com/jcpeterson/choices13k/refs/heads/main/c13k_problems.json", orient='index')

In [27]:
problems

,B,A
0,"[[0.9500000000000001, 21.0], [0.05, 23.0]]","[[0.9500000000000001, 26.0], [0.05, -1.0]]"
1,"[[0.75, -5.0], [0.25, 8.0]]","[[0.6000000000000001, 14.0], [0.4, -18.0]]"
2,"[[1.0, 1.0]]","[[0.5, 2.0], [0.5, 0.0]]"
3,"[[0.75, -31.0], [0.125, 86.5], [0.125, 87.5]]","[[0.05, 37.0], [0.9500000000000001, 8.0]]"
4,"[[0.25, -36.0], [0.375, 41.0], [0.1875, 43.0],...","[[1.0, 26.0], [0.0, 26.0]]"
...,...,...
14563,"[[0.199999999999999, 0.0], [0.8, 42.0]]","[[1.0, 30.0], [0.0, 30.0]]"
14564,"[[0.199999999999999, 7.0], [0.8, 18.0]]","[[0.5, 70.0], [0.5, -42.0]]"
14565,"[[0.6000000000000001, -34.0], [0.0125, 28.5], ...","[[0.4, 8.0], [0.6000000000000001, -17.0]]"
14566,"[[0.5, -12.0], [0.5, 45.0]]","[[0.5, 89.0], [0.5, -49.0]]"


In [5]:
choices_raw = choices.copy()

In [6]:
problems_raw = problems.copy()

## MODEL 1: LINEAR REGRESSION

After reading the papers, trying to understand where the dataset came from and what features should i use , i understood that this dataset is randomly selected , hence why they are different block numbers for different problems. I am not going to try to exactly analyze what cognitive behaviours are present in the dataset,nor will i try to directly rebuild the BEAST model, but what i will try is feature engineer features that somewhat represent the distinct anomalies with the sensitivity to expected return and the 4 tendecies:pessimism, bias toward equal weighting, sensitivity to payoff sign, and an effort
to minimize the probability of immediate regret

Lets first merge the choices and problems into one dataframe 

In [7]:
c13k_w_gambles = choices.join(problems, how="left")

In [31]:
c13k_w_gambles

,Problem,Feedback,n,Block,Ha,pHa,La,Hb,pHb,Lb,LotShapeB,LotNumB,Amb,Corr,bRate,bRate_std,B,A
0,1,True,15,2,26,0.95,-1,23,0.05,21,0,1,False,0,0.626667,0.384460,"[[0.9500000000000001, 21.0], [0.05, 23.0]]","[[0.9500000000000001, 26.0], [0.05, -1.0]]"
1,2,True,15,4,14,0.60,-18,8,0.25,-5,0,1,True,-1,0.493333,0.413118,"[[0.75, -5.0], [0.25, 8.0]]","[[0.6000000000000001, 14.0], [0.4, -18.0]]"
2,3,True,17,4,2,0.50,0,1,1.00,1,0,1,False,0,0.611765,0.432843,"[[1.0, 1.0]]","[[0.5, 2.0], [0.5, 0.0]]"
3,4,True,18,3,37,0.05,8,87,0.25,-31,1,2,False,0,0.222222,0.387383,"[[0.75, -31.0], [0.125, 86.5], [0.125, 87.5]]","[[0.05, 37.0], [0.9500000000000001, 8.0]]"
4,5,False,15,1,26,1.00,26,45,0.75,-36,2,5,False,0,0.586667,0.450185,"[[0.25, -36.0], [0.375, 41.0], [0.1875, 43.0],...","[[1.0, 26.0], [0.0, 26.0]]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14563,13002,True,15,3,30,1.00,30,42,0.80,0,0,1,True,0,0.367619,0.302731,"[[0.199999999999999, 0.0], [0.8, 42.0]]","[[1.0, 30.0], [0.0, 30.0]]"
14564,13003,True,15,5,70,0.50,-42,18,0.80,7,0,1,False,0,0.760000,0.364104,"[[0.199999999999999, 7.0], [0.8, 18.0]]","[[0.5, 70.0], [0.5, -42.0]]"
14565,13004,True,15,5,8,0.40,-17,31,0.40,-34,1,6,False,0,0.666667,0.367747,"[[0.6000000000000001, -34.0], [0.0125, 28.5], ...","[[0.4, 8.0], [0.6000000000000001, -17.0]]"
14566,13005,True,15,2,89,0.50,-49,45,0.50,-12,0,1,False,0,0.386667,0.381476,"[[0.5, -12.0], [0.5, 45.0]]","[[0.5, 89.0], [0.5, -49.0]]"


In [8]:
feature_table = c13k_w_gambles.copy()

feature_table["EV_A"] = feature_table["pHa"] * feature_table["Ha"] + (1 - feature_table["pHa"]) * feature_table["La"]
feature_table["EV_B"] = feature_table["pHb"] * feature_table["Hb"] + (1 - feature_table["pHb"]) * feature_table["Lb"]
feature_table["EV_diff"] = feature_table["EV_B"] - feature_table["EV_A"]


In [9]:
feature_table["Range_A"] = (feature_table["Ha"] - feature_table["La"]).abs()
feature_table["Range_B"] = (feature_table["Hb"] - feature_table["Lb"]).abs()
feature_table["Range_diff"] = feature_table["Range_B"] - feature_table["Range_A"]


In [10]:
feature_table["Worst_A"] = feature_table[["Ha", "La"]].min(axis=1)
feature_table["Worst_B"] = feature_table[["Hb", "Lb"]].min(axis=1)
feature_table["Worst_diff"] = feature_table["Worst_B"] - feature_table["Worst_A"]


In [11]:
feature_table["Amb"] = feature_table["Amb"].astype(int)
feature_table["Feedback"] = feature_table["Feedback"].astype(int)
feature_table["Block"] = feature_table["Block"].astype(int)
feature_table["Corr"] = feature_table["Corr"].astype(int)
feature_table["Feedback_Block"] = feature_table["Feedback"] * feature_table["Block"]


In [12]:
feature_cols = [
    "EV_diff",
    "Range_diff",
    "Worst_diff",
    "Amb",
    # "Feedback",
    # "Block",
    "Feedback_Block",
    "Corr",
]

X = feature_table[feature_cols]
y = feature_table["bRate"]


In [13]:
X

,EV_diff,Range_diff,Worst_diff,Amb,Feedback_Block,Corr
0,-3.55,-25,22,0,2,0
1,-2.95,-19,13,1,4,-1
2,0.00,-2,1,0,4,0
3,-10.95,89,-39,0,3,0
4,-1.25,81,-62,0,0,0
...,...,...,...,...,...,...
14563,3.60,42,-30,1,3,0
14564,1.80,-101,49,0,5,0
14565,-1.00,40,-17,0,5,0
14566,-3.50,-81,37,0,2,0


In [14]:
from sklearn.preprocessing import StandardScaler

In [15]:
scale_cols = ["EV_diff", "Range_diff", "Worst_diff"]

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[scale_cols] = scaler.fit_transform(X_scaled[scale_cols])

In [16]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_scaled, y)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)


In [18]:
lr = LinearRegression()
lr.fit(X_train, y_train)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [19]:
y_pred = lr.predict(X_test)


In [20]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Linear Regression MAE:  {mae:.4f}")
print(f"Linear Regression RMSE: {rmse:.4f}")
print(f"Linear Regression R²:   {r2:.4f}")


Linear Regression MAE:  0.1337
Linear Regression RMSE: 0.1643
Linear Regression R²:   0.4484


In [21]:
from sklearn.model_selection import cross_val_score

cv_mae = -cross_val_score(
    lr,
    X_scaled,
    y,
    scoring="neg_mean_absolute_error",
    cv=5
)

print(f"CV MAE: mean={cv_mae.mean():.4f}, std={cv_mae.std():.4f}")


CV MAE: mean=0.1332, std=0.0019


In [22]:
coef_table = (
    pd.Series(lr.coef_, index=X.columns)
      .sort_values(key=abs, ascending=False)
)

coef_table


EV_diff           0.125298
Worst_diff        0.069573
Amb               0.047099
Range_diff        0.032113
Corr              0.002382
Feedback_Block    0.002031
dtype: float64

In [23]:
results_linear = {
    "model": "Linear Regression",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "CV_MAE_mean": cv_mae.mean(),
    "CV_MAE_std": cv_mae.std()
}


In [24]:
results_linear

{'model': 'Linear Regression',
 'MAE': 0.1336768183408819,
 'RMSE': np.float64(0.16433025633154477),
 'R2': 0.44840794727121114,
 'CV_MAE_mean': np.float64(0.13321764876787778),
 'CV_MAE_std': np.float64(0.0019435749849000292)}

I will stop here since "Human choice behavior contains strong nonlinearities and interactions that linear models cannot capture."

## Random forest with extra features

In [25]:
rf_features = [
    "EV_diff",
    "Range_diff",
    "Worst_diff",
    "EV_A",
    "EV_B",
    "Range_A",
    "Range_B",
    "Worst_A",
    "Worst_B",
    "Amb",
    "Feedback",
    "Block",
    "Feedback_Block",
    "Corr",
    "LotNumB",
    "LotShapeB",
]

X_rf = feature_table[rf_features]
y = feature_table["bRate"]


In [28]:
from sklearn.model_selection import train_test_split

X_train_rf, X_test_rf, y_train, y_test = train_test_split(
    X_rf,
    y,
    test_size=0.2,
    random_state=42
)


In [29]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_rf, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred_rf = rf.predict(X_test_rf)

mae = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2 = r2_score(y_test, y_pred_rf)

print(f"RF Test MAE:  {mae:.4f}")
print(f"RF Test RMSE: {rmse:.4f}")
print(f"RF Test R²:   {r2:.4f}")


RF Test MAE:  0.0853
RF Test RMSE: 0.1100
RF Test R²:   0.7530


In [31]:
from sklearn.model_selection import cross_val_score

cv_mae_rf = -cross_val_score(
    rf,
    X_rf,
    y,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)

print(f"RF CV MAE: mean={cv_mae_rf.mean():.4f}, std={cv_mae_rf.std():.4f}")


RF CV MAE: mean=0.0873, std=0.0014


In [87]:
rf_importance = (
    pd.Series(rf.feature_importances_, index=X_rf.columns)
      .sort_values(ascending=False)
)

rf_importance.head(10)


EV_diff       0.483935
Worst_diff    0.133682
Range_B       0.068319
Worst_B       0.049010
Range_A       0.043872
Worst_A       0.043680
Range_diff    0.041844
EV_A          0.040227
Amb           0.024467
LotNumB       0.021506
dtype: float64

## Hyperparameter tuning

In [88]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


In [94]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_dist = {
    "n_estimators": [200, 300],
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", 0.5, 0.7, 1.0],
    "bootstrap": [True],
}


In [99]:
search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=40,  # increase to 80
    scoring="neg_mean_absolute_error",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

search.fit(X_rf, y)

best_rf = search.best_estimator_
print("Best params:", search.best_params_)
print(f"Best CV MAE: {-search.best_score_:.4f}")


Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 40, 'bootstrap': True}
Best CV MAE: 0.0860


In [100]:
best_rf.fit(X_train_rf, y_train)
y_pred = best_rf.predict(X_test_rf)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Tuned RF Test MAE:  {mae:.4f}")
print(f"Tuned RF Test RMSE: {rmse:.4f}")
print(f"Tuned RF Test R²:   {r2:.4f}")


Tuned RF Test MAE:  0.0838
Tuned RF Test RMSE: 0.1074
Tuned RF Test R²:   0.7644


## RF regression using XGBOOST

In [32]:
X_xgb = X_rf.copy()
y = feature_table["bRate"].copy()


In [33]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=4,          
    verbosity=1
)

xgb.fit(X_train_rf, y_train)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [34]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred_xgb = xgb.predict(X_test_rf)

mae = mean_absolute_error(y_test, y_pred_xgb)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2 = r2_score(y_test, y_pred_xgb)

print(f"XGB Test MAE:  {mae:.4f}")
print(f"XGB Test RMSE: {rmse:.4f}")
print(f"XGB Test R²:   {r2:.4f}")


XGB Test MAE:  0.0803
XGB Test RMSE: 0.1025
XGB Test R²:   0.7855


In [40]:
from sklearn.model_selection import cross_val_score

cv_mae_xgb = -cross_val_score(
    xgb,
    X_xgb,
    y,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=1   # avoid nested parallelism
)

print(f"XGB CV MAE: mean={cv_mae_xgb.mean():.4f}, std={cv_mae_xgb.std():.4f}")


XGB CV MAE: mean=0.0821, std=0.0010


In [75]:
import pandas as pd

xgb_importance = (
    pd.Series(xgb.feature_importances_, index=X_xgb.columns)
      .sort_values(ascending=False)
)

xgb_importance.head(15)


EV_diff       0.262407
Amb           0.083582
Worst_diff    0.081262
Worst_A       0.073309
Range_A       0.067749
Worst_B       0.065830
Range_B       0.061831
EV_A          0.061566
LotNumB       0.052233
LotShapeB     0.050483
Range_diff    0.042927
EV_B          0.040168
Feedback      0.019536
Corr          0.013192
Block         0.012526
dtype: float32

In [76]:
import pandas as pd


booster = xgb.get_booster()

gain = booster.get_score(importance_type="gain")

gain_imp = (pd.Series(gain, name="gain")
            .sort_values(ascending=False)
            .reset_index()
            .rename(columns={"index": "feature"}))

print(gain_imp.head(20))


           feature      gain
0          EV_diff  0.604999
1              Amb  0.192705
2       Worst_diff  0.187354
3          Worst_A  0.169019
4          Range_A  0.156200
5          Worst_B  0.151775
6          Range_B  0.142556
7             EV_A  0.141945
8          LotNumB  0.120428
9        LotShapeB  0.116392
10      Range_diff  0.098971
11            EV_B  0.092611
12        Feedback  0.045042
13            Corr  0.030416
14           Block  0.028880
15  Feedback_Block  0.026279


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, make_scorer

mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

perm = permutation_importance(
    xgb,
    X_test_rf,
    y_test,
    scoring=mae_scorer,
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

perm_imp = (
    pd.DataFrame({
        "feature": X_test_rf.columns,
        "mae_increase": -perm.importances_mean,  # flip sign
        "std": perm.importances_std
    })
    .sort_values("mae_increase", ascending=False)
)

perm_imp.head(15)


Changing the feature table

In [115]:
X_changed = X_xgb.copy()

In [116]:
X_changed = X_changed.drop(
    columns=["Feedback", "Block"]
)

In [ ]:
# X_changed = X_changed.drop(
#     columns=["Feedback", "Feedback_Block", "Block", "Corr"]
# )

In [117]:
y = feature_table["bRate"].copy()

In [118]:
from sklearn.model_selection import train_test_split

X_train_rf, X_test_rf, y_train, y_test = train_test_split(
    X_changed,
    y,
    test_size=0.2,
    random_state=42
)


In [149]:
from xgboost import XGBRegressor

xgb_changed = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    # objective="reg:logistic",
    # objective="reg:squarederror",
    objective="reg:squaredlogerror",
    # objective="reg:absoluteerror",
    # objective="reg:pseudohubererror",
    # objective="binary:logistic",
    # objective="binary:logitraw",
    random_state=42,
    n_jobs=4,          
    verbosity=1
)

xgb_changed.fit(X_train_rf, y_train)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squaredlogerror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabet

In [145]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred_xgb = xgb_changed.predict(X_test_rf)

## if logit_raw
# y_pred_xgb = 1 / (1 + np.exp(-y_pred_xgb))

mae = mean_absolute_error(y_test, y_pred_xgb)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2 = r2_score(y_test, y_pred_xgb)

print(f"XGB Test MAE:  {mae:.4f}")
print(f"XGB Test RMSE: {rmse:.4f}")
print(f"XGB Test R²:   {r2:.4f}")


XGB Test MAE:  0.0794
XGB Test RMSE: 0.1012
XGB Test R²:   0.7910


In [95]:
from sklearn.model_selection import cross_val_score

cv_mae_xgb = -cross_val_score(
    xgb_changed,
    X_changed,
    y,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=1   
)

print(f"XGB CV MAE: mean={cv_mae_xgb.mean():.4f}, std={cv_mae_xgb.std():.4f}")


XGB CV MAE: mean=0.0808, std=0.0011


In [109]:
import pandas as pd

xgb_importance = (
    pd.Series(xgb_changed.feature_importances_, index=X_changed.columns)
      .sort_values(ascending=False)
)

xgb_importance.head(15)


EV_diff           0.246937
Worst_diff        0.123117
Amb               0.084987
Range_A           0.082683
Range_B           0.064735
EV_A              0.060829
Worst_A           0.059266
LotNumB           0.055753
LotShapeB         0.052553
Worst_B           0.051372
Range_diff        0.046518
EV_B              0.042185
Feedback_Block    0.016819
Corr              0.012245
dtype: float32

In [110]:
import pandas as pd


booster = xgb_changed.get_booster()

gain = booster.get_score(importance_type="gain")

gain_imp = (pd.Series(gain, name="gain")
            .sort_values(ascending=False)
            .reset_index()
            .rename(columns={"index": "feature"}))

print(gain_imp.head(20))


           feature      gain
0          EV_diff  0.542133
1       Worst_diff  0.270295
2              Amb  0.186582
3          Range_A  0.181525
4          Range_B  0.142122
5             EV_A  0.133546
6          Worst_A  0.130114
7          LotNumB  0.122402
8        LotShapeB  0.115377
9          Worst_B  0.112784
10      Range_diff  0.102126
11            EV_B  0.092613
12  Feedback_Block  0.036924
13            Corr  0.026883


In [111]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, make_scorer

mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

perm = permutation_importance(
    xgb_changed,
    X_test_rf,
    y_test,
    scoring=mae_scorer,
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

perm_imp = (
    pd.DataFrame({
        "feature": X_test_rf.columns,
        "mae_increase": -perm.importances_mean,  # flip sign
        "std": perm.importances_std
    })
    .sort_values("mae_increase", ascending=False)
)

perm_imp.head(15)


,feature,mae_increase,std
11,Corr,0.000010,0.000038
10,Feedback_Block,-0.001598,0.000171
13,LotShapeB,-0.002431,0.000323
4,EV_B,-0.003036,0.000370
12,LotNumB,-0.005848,0.000311
7,Worst_A,-0.008419,0.000646
9,Amb,-0.009265,0.000459
1,Range_diff,-0.012515,0.000818
5,Range_A,-0.013686,0.000470
3,EV_A,-0.013765,0.000782


## Hyperparameter tuning with optuna on the XBBOOST model

In [98]:
import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import KFold


In [99]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)


In [146]:
def objective(trial):
    # params = {
    #     "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
    #     "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
    #     "max_depth": trial.suggest_int("max_depth", 2, 10),
    #     "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 20.0, log=True),
    #     "subsample": trial.suggest_float("subsample", 0.6, 1.0),
    #     "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    #     "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    #     "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
    #     "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    # }
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 800, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 20.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.8, 0.8),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.8, 0.8),
        # "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        # "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        # "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    

    model = XGBRegressor(
        **params,
        objective="reg:squaredlogerror",
        # objective="reg:logistic",
        random_state=42,
        n_jobs=1,         
        verbosity=0
    )

    model.fit(X_train_rf, y_train)
    
    y_pred_xgb = model.predict(X_test_rf)
    mae = mean_absolute_error(y_test, y_pred_xgb)

    # mae = mean_absolute_error(y_test, y_pred_xgb)
    # cross_val_score returns NEGATIVE MAE for this scorer
    # cv_mae = -cross_val_score(
    #     model,
    #     X_rf,
    #     y,
    #     scoring="neg_mean_absolute_error",
    #     cv=cv,
    #     n_jobs=1
    # ).mean()

    return mae


First parameter ranges

In [ ]:
# import numpy as np
# import optuna
# import xgboost as xgb
# from sklearn.model_selection import KFold
# from sklearn.metrics import mean_absolute_error
# from optuna.samplers import TPESampler

# # Fixed CV splitter for reproducibility
# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# # Build DMatrix once (faster)
# X_np = X_changed.values
# y_np = y.values
# dall = xgb.DMatrix(X_np, label=y_np)

# # Precompute folds (list of (train_idx, valid_idx))
# folds = [(tr, va) for tr, va in kf.split(X_np, y_np)]

# def objective(trial):
#     params = {
#         "verbosity": 0,
#         "objective": "reg:squarederror",
#         "eval_metric": "mae",
#         "tree_method": "hist",
#         "seed": 42,

#         # Smarter search space (focused)
#         "max_depth": trial.suggest_int("max_depth", 2, 8),
#         "eta": trial.suggest_float("eta", 0.01, 0.08, log=True),
#         "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 30.0, log=True),
#         "subsample": trial.suggest_float("subsample", 0.6, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
#         "gamma": trial.suggest_float("gamma", 0.0, 5.0),
#         "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
#         "lambda": trial.suggest_float("lambda", 1e-3, 30.0, log=True),
#     }

#     num_boost_round = trial.suggest_int("num_boost_round", 500, 3000)

#     # Run CV using xgboost.cv (simple + robust)
#     cv_results = xgb.cv(
#         params=params,
#         dtrain=dall,
#         folds=folds,
#         num_boost_round=num_boost_round,
#         early_stopping_rounds=80,
#         verbose_eval=False
#     )

#     # Min of validation MAE over boosting rounds
#     return float(cv_results["test-mae-mean"].min())


Other parameters

In [ ]:
# import numpy as np
# import optuna
# import xgboost as xgb
# from sklearn.model_selection import KFold
# from sklearn.metrics import mean_absolute_error
# from optuna.samplers import TPESampler

# # Fixed CV splitter for reproducibility
# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# # Build DMatrix once (faster)
# X_np = X_rf.values
# y_np = y.values
# dall = xgb.DMatrix(X_np, label=y_np)

# # Precompute folds (list of (train_idx, valid_idx))
# folds = [(tr, va) for tr, va in kf.split(X_np, y_np)]
# def objective_refine(trial):
#     params = {
#         "verbosity": 0,
#         "objective": "reg:squarederror",
#         "eval_metric": "mae",
#         "tree_method": "hist",
#         "seed": 42,

#         # refine around depth=6
#         "max_depth": trial.suggest_int("max_depth", 4, 7),

#         # refine around eta≈0.021
#         "eta": trial.suggest_float("eta", 0.012, 0.035, log=True),

#         # broaden slightly upward to control overfit
#         "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 8.0, log=True),

#         # keep sampling in good region
#         "subsample": trial.suggest_float("subsample", 0.65, 0.95),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.95),

#         # allow gamma to matter more (sometimes helps)
#         "gamma": trial.suggest_float("gamma", 0.0, 0.5),

#         # allow meaningful regularization (your best was ~0)
#         "alpha": trial.suggest_float("alpha", 1e-8, 1e-1, log=True),
#         "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
#     }

#     # keep boosting rounds in the right ballpark, let early stopping decide
#     num_boost_round = trial.suggest_int("num_boost_round", 1200, 3500)

#     cv_results = xgb.cv(
#         params=params,
#         dtrain=dall,
#         folds=folds,
#         num_boost_round=num_boost_round,
#         early_stopping_rounds=80,
#         verbose_eval=False
#     )

#     return float(cv_results["test-mae-mean"].min())


In [147]:
from optuna.samplers import TPESampler

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="minimize", sampler=sampler)

study.optimize(objective, n_trials=100, n_jobs=4, show_progress_bar=True)

print("Best MAE:", study.best_value)
print("Best params:", study.best_params)


[I 2026-02-03 21:53:13,345] A new study created in memory with name: no-name-e3b3d3f2-2eec-4849-b577-a3799cc8050e


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-02-03 21:53:18,442] Trial 3 finished with value: 0.08484859721257783 and parameters: {'n_estimators': 873, 'learning_rate': 0.1471707565351741, 'max_depth': 6, 'min_child_weight': 1.7761645257473837, 'subsample': 0.8, 'colsample_bytree': 0.8}. Best is trial 3 with value: 0.08484859721257783.
[I 2026-02-03 21:53:19,269] Trial 0 finished with value: 0.08499672755250039 and parameters: {'n_estimators': 1137, 'learning_rate': 0.1543232891803364, 'max_depth': 6, 'min_child_weight': 15.401146776500143, 'subsample': 0.8, 'colsample_bytree': 0.8}. Best is trial 3 with value: 0.08484859721257783.
[I 2026-02-03 21:53:22,412] Trial 2 finished with value: 0.08356384281278896 and parameters: {'n_estimators': 1939, 'learning_rate': 0.11247554366035516, 'max_depth': 5, 'min_child_weight': 12.450611229500637, 'subsample': 0.8, 'colsample_bytree': 0.8}. Best is trial 2 with value: 0.08356384281278896.
[I 2026-02-03 21:53:24,748] Trial 1 finished with value: 0.08642949599620936 and parameters: {

In [160]:
from sklearn.model_selection import cross_validate
from scipy.stats import sem
best_params = study.best_params

best_xgb = XGBRegressor(
    **best_params,
    objective="reg:squaredlogerror",
    random_state=42,
    n_jobs=4,
    verbosity=0
)


scores = cross_validate(
    best_xgb,
    X_changed,
    y,
    scoring=['r2', 'neg_mean_absolute_error'],
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=1   
)

best_xgb.fit(X_train_rf, y_train)

y_pred = best_xgb.predict(X_test_rf)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
# if mae != study.best_value:
#     print("You have made a mistake!!!!")
print(f"Tuned XGB Test MAE:  {study.best_value:.4f}")
print(mae)
print('r2', np.mean(scores['test_r2']), sem(scores['test_r2']))
print('mean_absolute_error',
      np.mean(-scores['test_neg_mean_absolute_error']),
      sem(-scores['test_neg_mean_absolute_error']))
# print(f"Tuned XGB Test Crossvalidated MAE:  {cv_mae_xgb.mean():.4f}, std={cv_mae_xgb.std():.4f}")
print(f"Tuned XGB Test RMSE: {rmse:.4f}")
# print(f"Tuned XGB Test R²:   {r2:.4f}")


Tuned XGB Test MAE:  0.0792
0.07943618614369376
r2 0.7871219223589263 0.002959192195852848
mean_absolute_error 0.08119493356685219 0.0005397538207172701
Tuned XGB Test RMSE: 0.1010


## Lets run a different model, were a complete example is in the lectures, LightGBM

In [154]:
import lightgbm as lgb

lgb_reg = lgb.LGBMRegressor(force_col_wise=True)


In [156]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_changed,
    y,
    test_size=0.2,
    random_state=42
)


In [162]:
lgb_reg.fit(X_train, y_train)
preds = lgb_reg.predict(X_test)

[LightGBM] [Info] Total Bins 1591
[LightGBM] [Info] Number of data points in the train set: 11654, number of used features: 14
[LightGBM] [Info] Start training from score 0.518642


In [ ]:
def objective(trial):

    X_train, X_test, y_train, y_test = train_test_split(
        X_changed,
        y,
        test_size=0.2,
        random_state=42
    )

    dtrain = lgb.Dataset(X_train, label=y_train)
    dvalid = lgb.Dataset(X_test, label=y_test)

    param = {
        "objective": "regression",
        "metric": "mean_squared_error",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "num_leaves": trial.suggest_int("num_leaves", 40, 80),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "n_estimators": trial.suggest_int("n_estimators", 500, 1500),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 0, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }

    # Add a callback for pruning.
    pruning_callback = optuna.integration.LightGBMPruningCallback(
        trial, metric="l2", valid_name="valid")
    gbm = lgb.train(param, dtrain, valid_sets=[dvalid],
                    valid_names="valid",
                    callbacks=[pruning_callback])

    preds = gbm.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    return mae

In [165]:
SEED = 42

np.random.seed(SEED)

In [232]:
from datetime import datetime
now = datetime.now()
timestamp_string = now.strftime("%Y-%m-%d %H:%M:%S")

study_name = f"lightGBM-bc-{timestamp_string}" # Unique identifier of the study.

study = optuna.create_study(
    study_name=study_name,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=50),
)
study.optimize(objective, n_trials=100, timeout=600)

print("Best MAE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-02-03 23:21:08,631] A new study created in memory with name: lightGBM-bc-2026-02-03 23:21:08
[I 2026-02-03 23:21:31,951] Trial 0 finished with value: 0.08608057248806052 and parameters: {'num_leaves': 55, 'learning_rate': 0.09556428757689246, 'n_estimators': 1232, 'colsample_bytree': 0.759195090518222, 'subsample': 0.4936111842654619, 'subsample_freq': 1, 'min_child_samples': 10}. Best is trial 0 with value: 0.08608057248806052.
[I 2026-02-03 23:21:36,741] Trial 1 finished with value: 0.08326975417917906 and parameters: {'num_leaves': 75, 'learning_rate': 0.0641003510568888, 'n_estimators': 1208, 'colsample_bytree': 0.41235069657748147, 'subsample': 0.9819459112971965, 'subsample_freq': 9, 'min_child_samples': 25}. Best is trial 1 with value: 0.08326975417917906.
[I 2026-02-03 23:21:40,510] Trial 2 finished with value: 0.08054103937378541 and parameters: {'num_leaves': 47, 'learning_rate': 0.026506405886809047, 'n_estimators': 804, 'colsample_bytree': 0.7148538589793427, 'subsa

Best MAE: 0.08054103937378541
Best params: {'num_leaves': 47, 'learning_rate': 0.026506405886809047, 'n_estimators': 804, 'colsample_bytree': 0.7148538589793427, 'subsample': 0.6591670111852694, 'subsample_freq': 3, 'min_child_samples': 63}


In [233]:
print('Number of finished trials: ', len(study.trials))
print('Best trial:')
best_trial = study.best_trial

print('Mae: ', best_trial.value)
print('Params: ', study.best_params)

Number of finished trials:  100
Best trial:
Mae:  0.08054103937378541
Params:  {'num_leaves': 47, 'learning_rate': 0.026506405886809047, 'n_estimators': 804, 'colsample_bytree': 0.7148538589793427, 'subsample': 0.6591670111852694, 'subsample_freq': 3, 'min_child_samples': 63}


In [234]:
gbm = lgb.LGBMRegressor(**best_trial.params)
scores = cross_validate(
    gbm,
    X_changed,
    y,
    scoring=['r2', 'neg_mean_absolute_error'],
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=1   
)

print('r2', np.mean(scores['test_r2']), sem(scores['test_r2']))
print('MAE',
      np.mean(-scores['test_neg_mean_absolute_error']),
      sem(-scores['test_neg_mean_absolute_error'])
)

r2 0.7852076622236718 0.0028781166374264398
MAE 0.08134902194872598 0.0004380683534706671
